In [1]:
!pip install google-generativeai

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 38.7 MB/s eta 0:00:00m eta 0:00:010:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.4
    Uninstalling protobuf-6.33.4:
      Successfully uninstalled protobuf-6.33.4
  Attempting uninstall: grpcio-status━━━━━━━━━━━━━━━━━ 0/7 [protobuf]
    Found existing installation: grpcio-status 1.76.0m 0/7 [protobuf]
    Uninstalling grpcio-status-1.76.0:━━━━━━━━━━━━ 0/7 [protobuf]
      Successfully uninstalled grpcio-status-1.76.0 0/7 [protobuf]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [google-generativeai]0m 5/7 [google

In [21]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai.types import (
    GenerateContentConfig,
    GoogleSearch,
    Tool
)

load_dotenv()
# 1. Configuración del Cliente (Uso de variables de entorno por seguridad)
# Asegúrate de configurar tu API KEY en las variables de entorno o pasarla aquí.
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# 2. Definición del modelo y la configuración de búsqueda
MODEL_ID = "gemini-2.5-flash"

def buscar_con_grounding(pregunta):
    try:
        # 3. Ejecución de la consulta con la herramienta de Google Search
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=pregunta,
            config=GenerateContentConfig(
                tools=[Tool(google_search=GoogleSearch())]
            )
        )

        # 4. Impresión del texto principal de la respuesta
        print(f"\n--- RESPUESTA ---\n{response.text}")

        # 5. Procesamiento detallado de fuentes (Grounding Metadata)
        metadata = response.candidates[0].grounding_metadata
        
        if metadata and metadata.search_entry_point:
            print("\n--- FUENTES VERIFICADAS ---")
            # Iteramos sobre los fragmentos (chunks) que contienen información de la web
            if metadata.grounding_chunks:
                for chunk in metadata.grounding_chunks:
                    if chunk.web:
                        print(f"• {chunk.web.title}")
                        print(f"  Enlace: {chunk.web.uri}")
        else:
            print("\nNo se utilizaron fuentes externas específicas.")

    except Exception as e:
        print(f"Error al conectar con la API: {e}")

# Bloque de ejecución principal
if __name__ == "__main__":
    pregunta_usuario = "¿Cuándo es el próximo eclipse solar total en España?"
    buscar_con_grounding(pregunta_usuario)


--- RESPUESTA ---
El próximo eclipse solar total en España tendrá lugar el 12 de agosto de 2026. Este será el primer eclipse total de Sol visible en la península ibérica en más de un siglo. La franja de totalidad cruzará el país de oeste a este, abarcando gran parte de la mitad norte peninsular, incluyendo ciudades como La Coruña, León, Bilbao, Zaragoza y Valencia. El fenómeno ocurrirá durante el atardecer, lo que requerirá un lugar de observación con buena visibilidad hacia el oeste.

Posteriormente, habrá otro eclipse solar total el 2 de agosto de 2027, que será visible en una franja del sur de España, incluyendo ciudades como Cádiz, Málaga, Ceuta y Melilla. Además, un eclipse solar anular se podrá observar el 26 de enero de 2028 en el suroeste de la península.

--- FUENTES VERIFICADAS ---
• fundaciondescubre.es
  Enlace: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHwPhgn7fayhfmT3TikwH0XmwRXU0BbrCvB2xlK2myGU4sfTdzfTDbNMTJz1ytawWgWdAkCaK_C22xuCdhb4mC3v3P140XI

In [23]:
import time

pregunta_input = input("Introduce tu noticia o duda de actualidad: ")

# Retry logic for handling temporary service unavailability
max_retries = 3
retry_delay = 5  # seconds

for attempt in range(max_retries):
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=pregunta_input,
            config=GenerateContentConfig(
                tools=[Tool(google_search=GoogleSearch())]
            )
        )
        break  # Success, exit retry loop
    except Exception as e:
        if attempt < max_retries - 1:
            print(f"Attempt {attempt + 1} failed: {e}")
            print(f"Retrying in {retry_delay} seconds...")
            time.sleep(retry_delay)
        else:
            print(f"Failed after {max_retries} attempts: {e}")
            raise

print("\n=== RESPUESTA VERIFICADA ===")
print(response.text)

# Extraer y mostrar las fuentes
try:
    chunks = response.candidates[0].grounding_metadata.grounding_chunks
    if chunks:
        print("\n=== FUENTES ===")
        for i, chunk in enumerate(chunks, 1):
            titulo = chunk.web.title
            url = chunk.web.uri
            print(f"{i}. {titulo}\n   {url}")
    else:
        print("\n⚠️  Advertencia: Esta respuesta se basa en mi entrenamiento "
              "interno y no ha sido verificada en tiempo real.")
except AttributeError:
    print("\n⚠️  Advertencia: Esta respuesta se basa en mi entrenamiento "
          "interno y no ha sido verificada en tiempo real.")


=== RESPUESTA VERIFICADA ===
El primer equipo de fútbol matemáticamente descendido de las 5 grandes ligas europeas en la temporada 2025-2026 es el **Wolverhampton Wanderers** de la Premier League. Su descenso se confirmó el 20 de abril de 2026.

=== FUENTES ===
1. onefootball.com
   https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGTd8r1O5lu0Le2C44ZLZdbxmi-sXoh4r-LUFf4vGmzcuMbLwcVEFiiJ1yPKhN98zG9Ur9yTgQbzQtQ2br7-EViE6MsY9WHu-P8aHsoD4mAPJSiXyYzaiAn3x6SJrxeuomMk5RL45jfeRAv7As6l4GacZU0PHo-bimuhfpkE_Q5p76dAauPGSZLcQjEdFx_l0-TwXObLTpggT7BL07FqWiH6zVZpDLP9ydmPvau
2. ecuavisa.com
   https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEpLPlkJDaSsrJcEHb1mzsyg-rN8Sr1hrhX7gIa0wr30vcSiruHsrYsaRvHA7EmIKrwvwN2EgFu5D2tzRrSkrcr9D6NAP0lFeD6oAy7b8WRPxeZqOf-m_sBlW2NeItSXj-uFCafM0AH1M_xLYU5HV0R4iYH4tWLAK5mO8lG-v1L4BU3gyDwENhoHBoWBMqNaKrysj2wxMHz6Nh8AtZemKHfUDm6Qwet6UQB9nsLphi-Bd8=
3. onefootball.com
   https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZ